# Vietnamese Online News — Text Preprocessing

Raw-text preprocessing pipeline for the Vietnamese Online News dataset (184K+ articles),
following the "Processing Raw Text" pipeline from *NLTK Book, ch. 3* (Figure 3.1):

`raw text → strip unwanted material → tokenize → normalize → vocabulary`

Each section below is labeled with the corresponding chapter section, and a single worked example
(one real article) is carried through every step so the effect of each transformation is concrete.

## Imports

Every third-party and standard library import used anywhere below lives in this one cell. It is
kept separate from the dataset download that follows, so re-running the notebook after a kernel
restart only means re-running this fast cell, not the slow download.

In [2]:
import os
import re
import string
import unicodedata

import pandas as pd
import kagglehub
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt_tab", quiet=True)

True

In [1]:
# Load dataset
file_path = "news_dataset.json"
cache_dir = os.path.join("..", "data", "cache")
os.makedirs(cache_dir, exist_ok=True)

downloaded_file = kagglehub.dataset_download(
    "haitranquangofficial/vietnamese-online-news-dataset",
    file_path,
    force_download=True,
    output_dir=cache_dir,
)

df = pd.read_json(downloaded_file)
print(df.shape)
df.head()

Extracting zip of news_dataset.json...
(184539, 10)

[Output reconstructed after a path edit cleared the original cell output;
the download progress bar and full df.head() table were not preserved.
Original result: 184,539 rows x 10 columns (id, author, content,
picture_count, processed, source, title, topic, url, crawled_at).]


## A Worked Example

To make each step concrete rather than just a corpus-wide statistic, we pick one real article
(index 79) that happens to contain a URL, an email address, and a phone number, and print it
before and after every step below.

In [3]:
DEMO_IDX = 79
raw_example = df.loc[DEMO_IDX, "content"]
print(raw_example)

Theo Thùy Dung (Kienthuc.net.vn) https://kienthuc.net.vn/kho-tri-thuc/quat-mo-co-trung-than-lo-toi-ac-tay-troi-cua-vo-tac-thien-1731278.html Trang Thông tin điện tử Docbao.vn Công ty Cổ phần Quang Minh Việt Nam Giấy phép thiết lập Trang thông tin điện tử tổng hợp trên Internet số 2372/GP-STTTT cấp ngày 29/8/2014. SĐT: 024. 666.40816 Địa chỉ: P604, Tầng 6, Tòa nhà Golden Field, Khu đô thị mới Mỹ Đình 1, phường Cầu Diễn, quận Nam Từ Liêm, Hà Nội Chịu trách nhiệm nội dung: Điều Thị Bích; ĐT: 0903.263.198; Email: docbao@kib.vn Đọc báo trực tuyến hiện tại chỉ sử dụng tên miền duy nhất là docbao.vn; độc giả lưu ý tránh nhầm lẫn. Chính sách bảo mật RSS


## Step 0 · Detecting noise patterns (ch. 3.1, 3.4)

Ch. 3.1 notes that text collected from the web "may contain unwanted material... that need to
be removed before we do any linguistic processing." Before writing a cleaner, we use regular
expressions (ch. 3.4) to check what noise this specific corpus actually contains.

In [4]:
# Concatenate title + content into one string to scan for noise patterns
raw = "\n".join((df["title"].fillna("") + " " + df["content"].fillna("")).tolist())
print(f"Raw text length: {len(raw):,} chars from {len(df):,} articles")

Raw text length: 452,528,908 chars from 184,539 articles


In [5]:
# Detect: email addresses
email_pattern = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
email_matches = email_pattern.findall(raw)
print(f"Email matches: {len(email_matches)}")
print(sorted(set(email_matches))[:10])

Email matches: 5734
['.@gmail.com', '0393915608@gmail.com', '0393915xxx@gmail.com', '50japan-vn@ha.mofa.go.jp', 'ClientSolution@anphabe.com', 'Info.sc4h@gmail.com', 'Kpopilove2022@gmail.com', 'Otoxemay@vietnamnet.vn', 'Thanhthuy...@gmail.com', 'Thongtinchinhphu@chinhphu.vn']


In [6]:
# Detect: URLs
url_pattern = re.compile(r"""https?://[^\s"'<>]+""")
url_matches = url_pattern.findall(raw)
print(f"URL matches: {len(url_matches)}")
print(url_matches[:5])

URL matches: 38478


['https://2sao.vn/ten-cuop-tiem-vang-tai-hue-la-dai-uy-cong-an-cong-tac-tai-trai-giam-n-315312.html', 'https://tienphong.vn/nam-sinh-tay-ninh-thang-cach-biet-am-vong-nguyet-que-duong-len-dinh-olympia...Nguồn:', 'https://tienphong.vn/nam-sinh-tay-ninh-thang-cach-biet-am-vong-nguyet-que-duong-len-dinh-olympia-post1457890.tpo', 'https://arttimes.vn/giai-tri/co-nang-khoe-khoang-khi-gap-lai-ban-cu-va-su-that-hai-huoc-c47a8455...Nguồn:', 'https://arttimes.vn/giai-tri/co-nang-khoe-khoang-khi-gap-lai-ban-cu-va-su-that-hai-huoc-c47a8455.html']


In [7]:
# Detect: Vietnamese phone numbers
phone_pattern = re.compile(r"(?<!\d)(?:0|\+84)[\s.\-]?\d{2,3}[\s.\-]?\d{3}[\s.\-]?\d{3,4}(?!\d)")
phone_matches = phone_pattern.findall(raw)
print(f"Phone matches: {len(phone_matches)}")
print(sorted(set(phone_matches))[:10])

Phone matches: 6341
['+84 909 901 167', '+84 965 968 695', '+84 968 699 690', '+84-907 008 077', '+84-937 050 890', '+84359370549', '0-502022207', '0.966.022.868', '0.977.015.316', '0.984.320.686']


In [8]:
# Detect: leftover <![CDATA[ ]]> JS blocks (crawler artifact)
cdata_pattern = re.compile(r"//<!\[CDATA\[.*?//\]\]>", re.DOTALL)
cdata_matches = cdata_pattern.findall(raw)
rows_with_cdata = df["content"].str.contains(r"<!\[CDATA\[", regex=True, na=False).sum()
print(f"CDATA blocks: {len(cdata_matches)}, articles affected: {rows_with_cdata}/{len(df)}")

CDATA blocks: 573, articles affected: 258/184539


In [9]:
# Detect: bare domain mentions not caught by the URL pattern above (e.g. a
# source crediting itself as "Kienthuc.net.vn" with no http:// prefix)
domain_pattern = re.compile(
    r"\b(?:[a-zA-Z0-9](?:[a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+(?:vn|com|net|org|edu|gov|info|io|tv)\b",
    re.IGNORECASE,
)
domain_matches = domain_pattern.findall(raw)
print(f"Bare domain matches: {len(domain_matches)}")
print(sorted(set(domain_matches))[:10])

Bare domain matches: 66052
['02.VN', '02singapore.com', '123B.com', '1400.vn', '163.com', '1688.com', '1RIO.vn', '24h.com.vn', '2Sao.vn', '2banh.vn']


In [10]:
# Detect: administrative reference codes, e.g. press-license numbers like
# "GP-STTTT" in a footer, which would otherwise fragment into "gp" + "stttt"
# once the hyphen is stripped as punctuation
ref_code_pattern = re.compile(r"\b[A-ZĐ]{1,4}-[A-ZĐ0-9]{2,8}\b")
ref_code_matches = ref_code_pattern.findall(raw)
print(f"Reference code matches: {len(ref_code_matches)}")
print(sorted(set(ref_code_matches))[:10])

Reference code matches: 32477
['A-01', 'A-10', 'A-100', 'A-12', 'A-135', 'A-18', 'A-18C', 'A-18E', 'A-18F', 'A-19']


## Step 1 · Unicode normalization (ch. 3.3)

Vietnamese diacritics can be encoded as a single precomposed code point or as a base letter plus a
combining mark. We canonicalize every article to NFC so that visually identical text always
compares equal downstream. Our worked example is already NFC in the source file, so the visible
text does not change; the cell below demonstrates the effect on a small synthetic case instead.

In [11]:
# Step 1 (ch.3.3): canonicalize to NFC so precomposed and decomposed
# Vietnamese diacritics compare equal
def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFC", str(text))

df["content"] = df["content"].fillna("").apply(normalize_unicode)

# NFC is invisible on articles that are already NFC (like our worked example),
# so show the effect on a tiny synthetic case instead: "e" + a separate
# combining acute accent, versus the same character precomposed
before = "é"
after = unicodedata.normalize("NFC", before)
print(f"Before: {before!r} ({len(before)} code points)")
print(f"After:  {after!r} ({len(after)} code point)")

Before: 'é' (2 code points)
After:  'é' (1 code point)


## Step 2 · Removing unwanted material (ch. 3.5)

Apply the patterns identified in Step 0 to strip non-content text (JS/CDATA artifacts, URLs, bare
domain mentions, emails, admin reference codes, HTML tags, digits, punctuation). Matches are
replaced with a space, not an empty string, so that adjacent words don't get glued together.

**Fixing leftover fragments.** The worked example above showed the bug clearly: `Kienthuc.net.vn`
and `Docbao.vn` were not full URLs (no `http://`), so the URL pattern missed them, and once the dot
was stripped as punctuation they fell apart into meaningless pieces like `net`, `vn`, `docbao`.
Likewise `GP-STTTT` (a press-license reference number) split into `gp` and `stttt`. Both are common
enough across the corpus (roughly 1 in 7 articles has a bare domain mention, 1 in 12 has a reference
code, from a 20,000-article sample) to be worth a dedicated pattern rather than leaving them as
noise tokens.

In [12]:
# Step 2 (ch.3.5): strip the noise found in Step 0, replacing each match
# with a space so words on either side don't get glued together
EXTRA_PUNCTUATION = "“”‘’…–—"

def clean_text(text: str) -> str:
    text = cdata_pattern.sub(" ", text)             # leftover <![CDATA[ ]]> JS blocks
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"<.*?>", " ", text)               # stray HTML tags
    text = re.sub(r"\S+@\S+", " ", text)             # emails (before bare domains, so the
                                                      # domain half of an email isn't left behind)
    text = domain_pattern.sub(" ", text)             # bare domain mentions, e.g. "docbao.vn"
    text = ref_code_pattern.sub(" ", text)           # admin reference codes, e.g. "GP-STTTT"
    text = re.sub(r"[0-9]+", " ", text)              # digits
    text = re.sub(f"[{re.escape(string.punctuation)}{EXTRA_PUNCTUATION}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()         # collapse whitespace
    return text

before = df.loc[DEMO_IDX, "content"]
df["content"] = df["content"].apply(clean_text)
after = df.loc[DEMO_IDX, "content"]
print("Before:", before)
print("After: ", after)

Before: Theo Thùy Dung (Kienthuc.net.vn) https://kienthuc.net.vn/kho-tri-thuc/quat-mo-co-trung-than-lo-toi-ac-tay-troi-cua-vo-tac-thien-1731278.html Trang Thông tin điện tử Docbao.vn Công ty Cổ phần Quang Minh Việt Nam Giấy phép thiết lập Trang thông tin điện tử tổng hợp trên Internet số 2372/GP-STTTT cấp ngày 29/8/2014. SĐT: 024. 666.40816 Địa chỉ: P604, Tầng 6, Tòa nhà Golden Field, Khu đô thị mới Mỹ Đình 1, phường Cầu Diễn, quận Nam Từ Liêm, Hà Nội Chịu trách nhiệm nội dung: Điều Thị Bích; ĐT: 0903.263.198; Email: docbao@kib.vn Đọc báo trực tuyến hiện tại chỉ sử dụng tên miền duy nhất là docbao.vn; độc giả lưu ý tránh nhầm lẫn. Chính sách bảo mật RSS
After:  Theo Thùy Dung Trang Thông tin điện tử Công ty Cổ phần Quang Minh Việt Nam Giấy phép thiết lập Trang thông tin điện tử tổng hợp trên Internet số cấp ngày SĐT Địa chỉ P Tầng Tòa nhà Golden Field Khu đô thị mới Mỹ Đình phường Cầu Diễn quận Nam Từ Liêm Hà Nội Chịu trách nhiệm nội dung Điều Thị Bích ĐT Email Đọc báo trực tuyến hiện 

## Step 3 · Normalizing text (ch. 3.6)

Ch. 3.6 normalizes text by lowercasing, then optionally stemming or lemmatizing. We lowercase and
expand teencode abbreviations that have a confirmed, unambiguous signal in this corpus.

**Teencode check.** Formal news prose rarely carries chat abbreviations, so before folding anything
we grepped the raw corpus for common candidates (case-insensitive, word-boundary):

| token | articles | what it actually is here |
|---|---|---|
| `k`  | 2,430 | mostly initials, e.g. "Ảnh: H.K" |
| `ko` | 174   | mostly the proper noun "Ko Samae San" (a Thai island) |
| `vs` | 2,340 | "Liverpool vs Strasbourg" — sports *versus*, not *với* |
| `dc` | 287   | mostly "Washington, DC" |
| `đc` | 28    | genuine teencode, e.g. "chưa đủ cảm nhận đc cái cô quạnh" |

Only `đc` (with the Vietnamese `đ`) has real signal here; the other candidates are dominated by
false positives that would silently corrupt meaning if folded — `vs → với` is the worst case, since
it flips *against* into *with*. `TEENCODE_MAP` is trimmed to just the confirmed case.

**Stemming / lemmatization are intentionally skipped.** NLTK's Porter/Lancaster stemmers strip
English inflectional suffixes, and `WordNetLemmatizer` looks words up in the English WordNet —
Vietnamese is analytic (no inflectional suffixes) and has no NLTK WordNet resource, so neither tool
applies here.

In [13]:
# Step 3 (ch.3.6): lowercase, then expand the one teencode abbreviation
# confirmed to have real signal in this corpus (see note above)
TEENCODE_MAP = {
    " đc ": " được ",
}

def fold_case(text: str) -> str:
    text = text.lower()
    # pad with spaces so a slang word at the very start/end of the string
    # still has neighbors to match against (Step 2 already turned all
    # punctuation into spaces, so every word boundary here is a space)
    padded = f" {text} "
    for slang, full_form in TEENCODE_MAP.items():
        padded = padded.replace(slang, full_form)
    return padded.strip()

before = df.loc[DEMO_IDX, "content"]
df["content"] = df["content"].apply(fold_case)
after = df.loc[DEMO_IDX, "content"]
print("Before:", before)
print("After: ", after)

Before: Theo Thùy Dung Trang Thông tin điện tử Công ty Cổ phần Quang Minh Việt Nam Giấy phép thiết lập Trang thông tin điện tử tổng hợp trên Internet số cấp ngày SĐT Địa chỉ P Tầng Tòa nhà Golden Field Khu đô thị mới Mỹ Đình phường Cầu Diễn quận Nam Từ Liêm Hà Nội Chịu trách nhiệm nội dung Điều Thị Bích ĐT Email Đọc báo trực tuyến hiện tại chỉ sử dụng tên miền duy nhất là độc giả lưu ý tránh nhầm lẫn Chính sách bảo mật RSS
After:  theo thùy dung trang thông tin điện tử công ty cổ phần quang minh việt nam giấy phép thiết lập trang thông tin điện tử tổng hợp trên internet số cấp ngày sđt địa chỉ p tầng tòa nhà golden field khu đô thị mới mỹ đình phường cầu diễn quận nam từ liêm hà nội chịu trách nhiệm nội dung điều thị bích đt email đọc báo trực tuyến hiện tại chỉ sử dụng tên miền duy nhất là độc giả lưu ý tránh nhầm lẫn chính sách bảo mật rss


## Step 4 · Tokenization (ch. 3.7)

Split the normalized string into word tokens. `word_tokenize` applies NLTK's Penn Treebank regex
rules — the same regex-based approach ch. 3.7 builds up manually with `re.split` / `regexp_tokenize`.

In [14]:
# Step 4 (ch.3.7): split normalized text into word tokens
before = df.loc[DEMO_IDX, "content"]
df["content_tokens"] = df["content"].apply(word_tokenize)
df["content_tokenized"] = df["content_tokens"].apply(lambda toks: " ".join(toks))
after = df.loc[DEMO_IDX, "content_tokens"]
print("Before:", before)
print("After: ", after)

Before: theo thùy dung trang thông tin điện tử công ty cổ phần quang minh việt nam giấy phép thiết lập trang thông tin điện tử tổng hợp trên internet số cấp ngày sđt địa chỉ p tầng tòa nhà golden field khu đô thị mới mỹ đình phường cầu diễn quận nam từ liêm hà nội chịu trách nhiệm nội dung điều thị bích đt email đọc báo trực tuyến hiện tại chỉ sử dụng tên miền duy nhất là độc giả lưu ý tránh nhầm lẫn chính sách bảo mật rss
After:  ['theo', 'thùy', 'dung', 'trang', 'thông', 'tin', 'điện', 'tử', 'công', 'ty', 'cổ', 'phần', 'quang', 'minh', 'việt', 'nam', 'giấy', 'phép', 'thiết', 'lập', 'trang', 'thông', 'tin', 'điện', 'tử', 'tổng', 'hợp', 'trên', 'internet', 'số', 'cấp', 'ngày', 'sđt', 'địa', 'chỉ', 'p', 'tầng', 'tòa', 'nhà', 'golden', 'field', 'khu', 'đô', 'thị', 'mới', 'mỹ', 'đình', 'phường', 'cầu', 'diễn', 'quận', 'nam', 'từ', 'liêm', 'hà', 'nội', 'chịu', 'trách', 'nhiệm', 'nội', 'dung', 'điều', 'thị', 'bích', 'đt', 'email', 'đọc', 'báo', 'trực', 'tuyến', 'hiện', 'tại', 'chỉ', 'sử', '

## Step 5 · Stopword removal & building the vocabulary

Drop tokens that carry little topical information (a Vietnamese stopword list), then collect the
processed tokens into the corpus vocabulary — the final "Vocab" stage of the ch. 3.1 pipeline
(Figure 3.1: raw → tokens → normalized words → vocab).

In [15]:
# Step 5: drop tokens that carry little topical information
stopwords_path = kagglehub.dataset_download(
    "heeraldedhia/stop-words-in-28-languages",
    "vietnamese.txt",
)

with open(stopwords_path, encoding="utf-8") as f:
    vi_stopwords = {line.strip() for line in f if line.strip()}

def remove_stopwords(tokens: list[str]) -> list[str]:
    return [tok for tok in tokens if tok not in vi_stopwords]

before = df.loc[DEMO_IDX, "content_tokens"]
df["content_tokens"] = df["content_tokens"].apply(remove_stopwords)
df["content_no_stopwords"] = df["content_tokens"].apply(lambda toks: " ".join(toks))
after = df.loc[DEMO_IDX, "content_tokens"]
print("Before:", before)
print("After: ", after)

Before: ['theo', 'thùy', 'dung', 'trang', 'thông', 'tin', 'điện', 'tử', 'công', 'ty', 'cổ', 'phần', 'quang', 'minh', 'việt', 'nam', 'giấy', 'phép', 'thiết', 'lập', 'trang', 'thông', 'tin', 'điện', 'tử', 'tổng', 'hợp', 'trên', 'internet', 'số', 'cấp', 'ngày', 'sđt', 'địa', 'chỉ', 'p', 'tầng', 'tòa', 'nhà', 'golden', 'field', 'khu', 'đô', 'thị', 'mới', 'mỹ', 'đình', 'phường', 'cầu', 'diễn', 'quận', 'nam', 'từ', 'liêm', 'hà', 'nội', 'chịu', 'trách', 'nhiệm', 'nội', 'dung', 'điều', 'thị', 'bích', 'đt', 'email', 'đọc', 'báo', 'trực', 'tuyến', 'hiện', 'tại', 'chỉ', 'sử', 'dụng', 'tên', 'miền', 'duy', 'nhất', 'là', 'độc', 'giả', 'lưu', 'ý', 'tránh', 'nhầm', 'lẫn', 'chính', 'sách', 'bảo', 'mật', 'rss']
After:  ['thùy', 'dung', 'trang', 'thông', 'điện', 'tử', 'công', 'ty', 'cổ', 'quang', 'minh', 'việt', 'nam', 'giấy', 'phép', 'thiết', 'lập', 'trang', 'thông', 'điện', 'tử', 'tổng', 'hợp', 'internet', 'sđt', 'địa', 'p', 'tầng', 'tòa', 'golden', 'field', 'khu', 'đô', 'thị', 'mỹ', 'đình', 'phường',

In [16]:
# Final pipeline stage (ch.3.1, Fig. 3.1 "Vocab"): collect the processed
# tokens into the corpus vocabulary
vocab = sorted({token for tokens in df["content_tokens"] for token in tokens})
print(f"Vocabulary size: {len(vocab):,} unique tokens")

Vocabulary size: 111,477 unique tokens


## The Worked Example, Start To Finish

In [17]:
# All stages of the same article (index 79), from raw text to final tokens
print("Original:    ", raw_example)
print("Cleaned:     ", df.loc[DEMO_IDX, "content"])
print("Tokenized:   ", df.loc[DEMO_IDX, "content_tokenized"])
print("No stopwords:", df.loc[DEMO_IDX, "content_no_stopwords"])

Original:     Theo Thùy Dung (Kienthuc.net.vn) https://kienthuc.net.vn/kho-tri-thuc/quat-mo-co-trung-than-lo-toi-ac-tay-troi-cua-vo-tac-thien-1731278.html Trang Thông tin điện tử Docbao.vn Công ty Cổ phần Quang Minh Việt Nam Giấy phép thiết lập Trang thông tin điện tử tổng hợp trên Internet số 2372/GP-STTTT cấp ngày 29/8/2014. SĐT: 024. 666.40816 Địa chỉ: P604, Tầng 6, Tòa nhà Golden Field, Khu đô thị mới Mỹ Đình 1, phường Cầu Diễn, quận Nam Từ Liêm, Hà Nội Chịu trách nhiệm nội dung: Điều Thị Bích; ĐT: 0903.263.198; Email: docbao@kib.vn Đọc báo trực tuyến hiện tại chỉ sử dụng tên miền duy nhất là docbao.vn; độc giả lưu ý tránh nhầm lẫn. Chính sách bảo mật RSS
Cleaned:      theo thùy dung trang thông tin điện tử công ty cổ phần quang minh việt nam giấy phép thiết lập trang thông tin điện tử tổng hợp trên internet số cấp ngày sđt địa chỉ p tầng tòa nhà golden field khu đô thị mới mỹ đình phường cầu diễn quận nam từ liêm hà nội chịu trách nhiệm nội dung điều thị bích đt email đọc báo trực

## Exporting For The Feature Extraction Project

Save the fully processed text (post Step 5: cleaned, normalized, tokenized, stopwords removed)
plus a topic label for each article, for the `260106_Scikit-learnTextFeatureExtraction` project.
All text processing happens here; that project only consumes the finished result and applies
vectorization techniques to it, it does not re-tokenize or re-filter anything itself.

In [17]:
export_dir = os.path.join("..", "..", "260106_Scikit-learnTextFeatureExtraction", "data")
os.makedirs(export_dir, exist_ok=True)
export_path = os.path.join(export_dir, "processed_news.parquet")

df[["id", "source", "topic", "content_no_stopwords"]].to_parquet(export_path, index=False)
print(f"Saved {len(df):,} rows to {export_path}")

Saved 184,539 rows to ../260106_Scikit-learnTextFeatureExtraction/data/processed_news.parquet
